In [1]:
import numpy
import gensim

In [6]:
pairs_train=[] #These will be pairs of (source,target) i.e. (Finnish, English) words used to induce the matrix M
pairs_test=[]  #same but for testing, so we should make sure there is absolutely no overlap between the train and test data
               #let's do it so that not one word in the test is is seen in any capacity in the training data

import csv

def get_vectors(fname):
    """
    Read the pairs from the file `fname`
    """
    pairs=[]
    with open(fname) as f:
        r = csv.reader(f,delimiter="\t") #the file is a .tsv i.e. tab-separated-values
        for en_word,fi_word in r:
            #I will reverse the order here, go from Finnish as the source, to English as the target
            #That way it will be easier to check how this works using English as the target, which we all understand
            pairs.append((fi_word,en_word))
        return pairs

train_data=get_vectors("yacle.train.freq.5k.en-fi.tsv")
test_data=get_vectors("yacle.test.freq.2k.en-fi.tsv")
print(train_data[:10])
print(len(train_data))
print(test_data[:10])
print(len(test_data))



[('of', 'of'), ('että', 'to'), ('sisään', 'in'), ('varten', 'for'), ('on', 'is'), ('päällä', 'on'), ('että', 'that'), ('mennessä', 'by'), ('Tämä', 'this'), ('kanssa', 'with')]
5000
[('omistautuminen', 'dedication'), ('toiveet', 'desires'), ('hylätty', 'dismissed'), ('psyykkinen', 'psychic'), ('halkeamia', 'cracks'), ('laitokset', 'establishments'), ('tehokkuus', 'efficacy'), ('arvovalta', 'prestige'), ('kokaiini', 'cocaine'), ('kiihtyi', 'accelerated')]
2000


In [7]:
import numpy

def build_arrays(pairs,emb1,emb2,avoid=set()):
    """
    `pairs`: pairs of (fi,en) words
    `emb1`: source side (here Finnish) embeddings
    `emb2`: target side (here English) embeddings
    `avoid`: a set of words to avoid/ignore (will be used when building test data, to avoid train data)
    """
    vecs1,vecs2,filtered_pairs=[],[],[]  #vectors for source words, vectors for target words, and the word pairs themselves, i.e. three same-length lists
    for w1,w2 in pairs: #Go over all pairs that we got
        # check if both vectors are available, and none of the words is to be avoided
        if w1 in emb1 and w2 in emb2 and w1 not in avoid and w2 not in avoid:
            #passed!
            vecs1.append(emb1[w1]) #source-side embedding, the KeyedVectors object can be queried as if it was a dictionary, returns the embedding as 1-dim array
            vecs2.append(emb2[w2]) #target-side embeddings
            filtered_pairs.append((w1,w2)) #remember the pair
    #Now we vstack() which turns the lists of embeddings into 2-dim array
    return numpy.vstack(vecs1),numpy.vstack(vecs2),filtered_pairs

# Gather the train data first
array_train_fi,array_train_en,pairs_train=build_arrays(train_data,wv_emb_fi,wv_emb_en)
# Now build the set of all words seen in training, so we can avoid them when building the test set. Note that "|" is set union operator
everything_in_train=set(s for s,t in pairs_train)|set(t for s,t in pairs_train)
# Test data next, avoiding the words from the training data:
array_test_fi,array_test_en,pairs_test=build_arrays(test_data,wv_emb_fi,wv_emb_en,avoid=everything_in_train)

In [8]:
# Let's be super-sure there absolutely is no overlap of any kind!
print("Overlap between train pairs and test pairs:",len(set(pairs_train) & set(pairs_test))) # & is set intersection operator, intersection between train and test should be empty
src_train=set(src_w for src_w,tgt_w in pairs_train) #train source words
tgt_train=set(tgt_w for src_w,tgt_w in pairs_train) #train target words
src_test=set(src_w for src_w,tgt_w in pairs_test)   #test source words
tgt_test=set(tgt_w for src_w,tgt_w in pairs_test)   #test target words
print("Overlap between train fi words and test fi words:",len(src_train & src_test))
print("Overlap between train en words and test en words:",len(tgt_train & tgt_test))

Overlap between train pairs and test pairs: 0
Overlap between train fi words and test fi words: 0
Overlap between train en words and test en words: 0


In [9]:
# This code was written by GPT4, but in a bit of a twisted form, so I modified it
# to better correspond to the formulae in the lecture

def learn_transformation_matrix(source, target):
    # Compute the pseudo-inverse of the source matrix
    source_pseudo_inverse = numpy.linalg.pinv(source) # This implements (S^T S)^-1 S^T  needed in the least-squares formula in the lecture slides
    # Compute the transformation matrix M using least squares method
    M = numpy.matmul(source_pseudo_inverse,target)  #...and this multiplies by T from right completing the formula in the slides ... two lines(!)
    return M

# fi -> en matrix
M=learn_transformation_matrix(array_train_fi,array_train_en)

# Ha ha well that was easy

In [13]:
def top_k_accuracy(target, words, k):
  eng_words = [w for w, score in words] 
  nearest_k = eng_words[:k]

  if(nearest_k.count(target) > 0):
    return True
  else:
    return False

In [22]:
print(len(pairs_test))

1285


In [26]:
total = 0
num_of_true = 0
k = 10

for i,(w1,w2) in enumerate(pairs_test[:100]):
    total += 1
    nearest_neighbors=wv_emb_en.similar_by_vector(test_fi_transformed[i]) #lookup the nearest
    if top_k_accuracy(w2,nearest_neighbors,k):
        num_of_true += 1

print(f"top-{k} accuracy is: {num_of_true/total}")   

top-10 accuracy is: 0.39
